# Fourth LAB

### Select the Kernel

Click on "Select Kernel" on the Top Right

Choose "Python Environments..."

Then choose the one that looks like `.venv (Python 3.12.x) .venv/bin/python` - it should be marked as "Recommended" and have a big star next to it.

In [19]:
# imports

import os
from dotenv import load_dotenv
# from scraper import *
from IPython.display import Markdown, display
from huggingface_hub import login, InferenceClient
from tqdm.notebook import tqdm
from openai import OpenAI
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TextStreamer, TextIteratorStreamer
from threading import Thread
import torch
import gc
import gradio as gr

In [2]:
# Check for GPU / CPU
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

device = "cuda" if torch.cuda.is_available() else "cpu"

GPU: NVIDIA GeForce RTX 3050 Laptop GPU
CUDA: 12.1


### Connecting to Hugging Face

The next cell is where we load in the environment variables in your `.env` file and connect to Hugging Face.  

In [3]:
# Log in to HuggingFace - if you get a "Note" about Environment variable being set, ignore it

hf_token = os.environ['HF_TOKEN']
client = InferenceClient(api_key=hf_token) 
login(hf_token, add_to_git_credential = False)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
# Define model ID
model_id = "meta-llama/Llama-3.2-3B-Instruct"

### Load Model

In [5]:
# Load model
# This client sends requests to a remote inference provider.
client = InferenceClient(
    api_key=hf_token,
    provider="auto"
)

# Load model directly
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load model directly onto available hardware
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map=device,
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16,
)

print("Model loaded successfully!")
print("Memory footprint:", round(model.get_memory_footprint() / 1024**3, 2), "GB")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded successfully!
Memory footprint: 5.98 GB


In [ ]:
# Define system message
system_message = "You are a helpful assistant that responds in markdown"

In [ ]:
def model_call(prompt):
    
    # Defining prompt & message
    messages=[
            {"role": "system", "content": "You are a helpful assistant that responds in markdown"},
            {"role": "user", "content": prompt}
        ]

    # Streamer
    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True
        )
    
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(device)

    # Generation parameters
    generation_kwargs = {
            **inputs,
            "max_new_tokens": 250,
            "streamer": streamer
        }

    # Run model generation in a separate thread
    thread = Thread(
        target=model.generate,
        kwargs=generation_kwargs
    )

    thread.start()
    
    # Extract Response
    result = ""
    for new_text in streamer:
        result += new_text
        yield result
    
    thread.join()

### Testing out SLMs

In [25]:
# Gradio Ouptut
message_input = gr.Textbox(label="Your message:", info="Enter your query:", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=model_call,
    title="SLM", 
    inputs=[message_input], 
    outputs=[message_output], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [26]:
# Commands to clear GPU
del model, tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()